# Lezione 14: Casi d'uso reali — chiusura del corso

**Obiettivi di apprendimento:**
- Riconoscere applicazioni reali della scienza dei dati in ambito aziendale, sanitario e ambientale.
- Percorrere un flusso di lavoro completo (dati → pulizia → EDA → modello → report) su un caso reale.
- Integrare quanto appreso nelle 13 lezioni precedenti.

**Durata stimata:** 120 min

Questa è l'ultima lezione del corso. Non ci sono concetti nuovi da memorizzare — c'è un invito a vedere tutto ciò che già sai funzionare insieme, dall'inizio alla fine.

## La scienza dei dati è ovunque

- **Uber**: dati in tempo reale per assegnare autisti e fissare prezzi dinamici.
- **Toronto Transportation Commission**: combinando dati di tram, reclami e traffico, ha ridotto la congestione mensile da 4,75 h (2010) a 3 h (2014).
- **Monitoraggio ambientale**: imbarcazioni robotiche, boe e droni per sorvegliare le fioriture di cianobatteri.
- **Raccomandazione**: Amazon, Netflix, Spotify, Siri, wearable (Fitbit, Apple Watch).
- **UPS (2013)**: ottimizzazione delle rotte con dati di autisti, veicoli e clienti.
- **Sanità**: analitica predittiva per raccomandare esami/trattamenti; NorthShore University HealthSystem, primo fornitore degli USA con adozione completa delle Cartelle Cliniche Elettroniche.
- **Previsione dei disastri**: l'Università di Warwick combina social media e stazioni meteorologiche per tracciare alluvioni/uragani.

**Caso Netflix / House of Cards**: hanno analizzato dati di visione, pause, riavvolgimenti, ricerche di attore/regista e il successo della versione britannica originale, per dare il via libera al remake prima di girare un solo episodio. Netflix sapeva cosa voleva la gente prima che la gente stessa lo sapesse.

## Il viaggio di Lila: il caso integrativo

Lila, con formazione in economia, ottiene il suo primo posto come data scientist junior nel retail. Il suo compito: trovare pattern e anomalie nei dati dei clienti. Il suo processo:

1. **Selezione e ottenimento del dataset** da più fonti.
2. **Comprensione e pulizia** con Python e SQL: nulli, duplicati, outlier.
3. **EDA**: statistiche riassuntive, istogrammi, diagrammi a dispersione.
4. **Feature engineering**: ad esempio, calcolare l'importo totale di acquisto.
5. **Analisi statistica e ML**: regressione prezzo-vendite, previsione della domanda, segmentazione.
6. **Presentazione**: notebook → report + presentazione per l'area eCommerce.
7. **Apprendimento continuo**: Lila continua a formarsi dopo questo progetto.

Questo è esattamente il processo a sei passi del capitolo 1 (chiarire, raccogliere, analizzare, riconoscere pattern, raccontare la storia, visualizzare), applicato a un caso reale. Percorriamolo noi stessi con il codice, dall'inizio alla fine.

## Il flusso di lavoro completo, con il codice

Come sostituto riproducibile e 100% locale di un caso reale di regressione (come quello di previsione dell'area bruciata da incendi boschivi), usiamo il dataset diabetes integrato in scikit-learn: stesso tipo di problema (predire un numero a partire da variabili numeriche), senza scaricare nulla di esterno.

### Passo 1 — Caricare i dati

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)

# load_diabetes() è integrato in scikit-learn: nessun download esterno.
# Ogni riga è un paziente; ogni colonna, una variabile clinica già normalizzata.
# L'obiettivo ("target") è una misura numerica di progressione del diabete dopo un anno.
datos = load_diabetes(as_frame=True)
df = datos.frame

print("Righe, colonne:", df.shape)
df.head()

### Passo 2 — Analisi Esplorativa dei Dati (EDA)

In [ ]:
# Statistiche riassuntive: media, deviazione, minimi e massimi di ogni colonna
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Istogramma: come si distribuisce la variabile obiettivo
axes[0].hist(df["target"], bins=20, color="steelblue")
axes[0].set_title("Distribuzione dell'obiettivo (progressione della malattia)")
axes[0].set_xlabel("target")

# Diagramma a dispersione: l'indice di massa corporea ha relazione con l'obiettivo?
axes[1].scatter(df["bmi"], df["target"], alpha=0.5, color="seagreen")
axes[1].set_title("Indice di massa corporea (bmi) vs. obiettivo")
axes[1].set_xlabel("bmi (normalizzato)")
axes[1].set_ylabel("target")

plt.tight_layout()
plt.show()

L'istogramma ci dice se la variabile obiettivo è asimmetrica (se lo fosse molto, come nel caso reale degli incendi boschivi, converrebbe applicare una trasformazione logaritmica prima di modellare). Il diagramma a dispersione ci dà un primo indizio visivo: sembra esserci una relazione positiva tra `bmi` e `target` — a un indice di massa corporea maggiore corrisponde una progressione della malattia maggiore.

### Passo 3 — Pulizia minima (simulando un problema reale di dati)

Il dataset è già pulito, quindi —come ha fatto Lila con dati reali di clienti— iniettiamo noi stessi un problema di dati mancanti per esercitarci su come si tratta.

In [ ]:
# Simuliamo che il 5% dei valori di "bmi" siano andati persi (ad esempio,
# un sensore che ha fallito nel registrare il dato di alcuni pazienti).
n_filas = len(df)
indices_nulos = np.random.choice(df.index, size=int(n_filas * 0.05), replace=False)
df.loc[indices_nulos, "bmi"] = np.nan

print("Valori nulli prima della pulizia:", df["bmi"].isna().sum())

# Imputiamo con la mediana: una tecnica semplice e ragionevole quando mancano pochi valori
df["bmi"] = df["bmi"].fillna(df["bmi"].median())

print("Valori nulli dopo la pulizia:", df["bmi"].isna().sum())

### Passo 4 — Ingegneria delle caratteristiche (feature engineering)

In [ ]:
# Creiamo una colonna derivata semplice: un "indice combinato" di rischio metabolico
# che somma bmi e pressione arteriosa media (bp). Non è una formula medica reale —
# è un esempio di come si costruisce una nuova caratteristica a partire da altre,
# proprio come Lila ha calcolato l'importo totale di acquisto a partire da colonne più basilari.
df["indice_riesgo"] = df["bmi"] + df["bp"]

df[["bmi", "bp", "indice_riesgo"]].head()

### Passo 5 — Addestrare e valutare un modello di regressione

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo = LinearRegression()
modelo.fit(X_train, y_train)
predicciones = modelo.predict(X_test)

mse = mean_squared_error(y_test, predicciones)
r2 = r2_score(y_test, predicciones)

print(f"Errore quadratico medio (MSE): {mse:.2f}")
print(f"R² (proporzione di varianza spiegata): {r2:.2f}")

Il `mean_squared_error` misura, in media, quanto sbaglia il modello (più basso è, meglio è). L'`r2_score` indica quale proporzione della variazione dell'obiettivo il modello spiega: 1.0 sarebbe una predizione perfetta, 0.0 equivarrebbe a non spiegare nulla meglio della media.

Possiamo confrontare questo modello lineare con un albero decisionale, per vedere se cattura meglio relazioni non lineari tra le variabili:

In [ ]:
arbol = DecisionTreeRegressor(max_depth=4, random_state=42)
arbol.fit(X_train, y_train)
predicciones_arbol = arbol.predict(X_test)

mse_arbol = mean_squared_error(y_test, predicciones_arbol)
r2_arbol = r2_score(y_test, predicciones_arbol)

print(f"Albero decisionale — MSE: {mse_arbol:.2f}, R²: {r2_arbol:.2f}")
print(f"Regressione lineare — MSE: {mse:.2f}, R²: {r2:.2f}")

### Passo 6 — Report finale

Su questo insieme di dati sul diabete, l'analisi mostra una relazione chiara e positiva tra l'indice di massa corporea e la progressione della malattia l'anno successivo, visibile sia nel diagramma a dispersione sia nel peso assegnatole dal modello di regressione lineare. Il modello addestrato spiega una parte rilevante ma non totale della variazione dell'obiettivo, il che è prevedibile con modelli semplici e un insieme ridotto di variabili cliniche. Prima di usare questo modello in un contesto reale servirebbe continuare a esplorare modelli e variabili aggiuntive, ma il flusso di lavoro applicato qui —dal caricamento dei dati alla valutazione— è lo stesso che useresti in qualsiasi problema di regressione con dati tabulari, che si tratti di predire la progressione di una malattia, i prezzi delle abitazioni o l'area bruciata da un incendio boschivo.

## Riepilogo della lezione e chiusura del corso

- La scienza dei dati è già dietro decisioni quotidiane: Uber, Netflix, UPS, gestione del traffico, sanità, previsione dei disastri.
- Il caso di Lila riassume, in un contesto reale, lo stesso processo a sei passi del capitolo 1.
- Un flusso di lavoro tecnico completo: caricare → EDA → pulire → feature engineering → modellare → valutare → report.
- Nessun modello è perfetto al primo colpo — il valore sta nel processo completo e nel comunicare onestamente i risultati (lezione 13).

Con questa lezione si chiude il percorso completo del corso: da "cos'è la scienza dei dati?" (lezione 1) fino a un flusso di lavoro reale dall'inizio alla fine. Il resto è pratica costante.

**Manuale:** [16 — Casi d'uso reali](../manual/16_casos_de_uso_reales_IT.md)

**Pratica (capstone):** [Lezione 14 — Pratica](../notebooks_practica/clase14_practica_casos_de_uso_reales_IT.ipynb)